In [2]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/datasets/RADAR-MDD/speech-features-for-prediction")
OUT_DIR = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/data/processed")

files = [
    "RADAR-MDD-Anna-Finch-RADAR-MDD-CIBER-s1-07-08-2025.csv",
    "RADAR-MDD-Anna-Finch-RADAR-MDD-IISPV-s1-07-08-2025.csv",
    "RADAR-MDD-Anna-Finch-RADAR-MDD-KCL-s1-07-08-2025.csv",
    "RADAR-MDD-Anna-Finch-RADAR-MDD-VUmc-s1-07-08-2025.csv",
]

dfs = []
for f in files:
    df = pd.read_csv(DATA_DIR / f)
    df["source_file"] = f
    dfs.append(df)

radar = pd.concat(dfs, ignore_index=True)

radar["Date"] = pd.to_datetime(radar["Date"], errors="coerce")
radar["Patient_id"] = radar["Patient_id"].astype(str).str.strip()

# keep rows with outcome present
radar = radar.dropna(subset=["Patient_id", "Date", "PHQ8"]).copy()

# optional: rename for convenience
radar = radar.rename(columns={
    "Patient_id": "participant_id",
    "Date": "recording_date",
    "PHQ8": "phq8_score"
})

print(radar.shape)
print(radar.head())

radar.to_csv(OUT_DIR / "radar_model_dataset_raw_features.csv", index=False)

(8515, 34)
                                File                        participant_id  \
2224  20201230_1100-scripted-1-1.wav  00d1d60a-15cf-481b-8264-91705b6f8d97   
2225  20200617_1600-unscripted-1.wav  00d1d60a-15cf-481b-8264-91705b6f8d97   
2226  20200812_1000-unscripted-1.wav  00d1d60a-15cf-481b-8264-91705b6f8d97   
2227  20200311_1200-scripted-1-1.wav  00d1d60a-15cf-481b-8264-91705b6f8d97   
2228  20201119_1200-unscripted-1.wav  00d1d60a-15cf-481b-8264-91705b6f8d97   

        Dataset      Language        Task recording_date   Age  Gender  \
2224  RADAR-MDD  English (UK)    Scripted     2020-12-30  61.0     1.0   
2225  RADAR-MDD  English (UK)  Unscripted     2020-06-17  61.0     1.0   
2226  RADAR-MDD  English (UK)  Unscripted     2020-08-12  61.0     1.0   
2227  RADAR-MDD  English (UK)    Scripted     2020-03-11  61.0     1.0   
2228  RADAR-MDD  English (UK)  Unscripted     2020-11-19  61.0     1.0   

      Education_Years  Height  ...  std_F1_Loc  mean_B1_Loc  std_B1_Loc  \


In [3]:
radar = radar.sort_values(["participant_id", "recording_date"]).reset_index(drop=True)
first_dates = radar.groupby("participant_id")["recording_date"].transform("min")
radar["days_since_first"] = (radar["recording_date"] - first_dates).dt.days